In [ ]:
import tiktoken
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

%matplotlib inline
sns.set_theme(style="whitegrid")

# --- Configuration ---
PROCESSED_DIR = Path("../data/processed")

# Cost per 1 Million Input Tokens (USD)
PRICING_1M = {
    "DeepSeek V3 (Cached)": 0.014,
    "DeepSeek V3 (Miss)": 0.140,
    "Gemini 1.5 Flash": 0.075,
    "GPT-4o-mini": 0.150,
    "GPT-4o": 2.500
}

# Context Window Limits
LIMITS = {
    "DeepSeek V3": 64000,
    "GPT-4o": 128000
}

In [ ]:
def count_tokens(text: str, model: str = "cl100k_base") -> int:
    """Counts tokens using tiktoken encoder."""
    try:
        enc = tiktoken.get_encoding(model)
        return len(enc.encode(text or ""))
    except Exception:
        return 0

def analyze_corpus(directory: Path) -> pd.DataFrame:
    """Scans markdown files and calculates token statistics."""
    data = []
    files = list(directory.glob("*.md"))
    
    print(f"🔍 Scanning {len(files)} documents...")
    
    for f in tqdm(files, desc="Tokenizing"):
        try:
            content = f.read_text(encoding="utf-8")
            data.append({
                "filename": f.name,
                "tokens": count_tokens(content),
                "chars": len(content)
            })
        except Exception as e:
            print(f"Skipping {f.name}: {e}")
            
    return pd.DataFrame(data)

In [ ]:
df = analyze_corpus(PROCESSED_DIR)

if not df.empty:
    print("\n📊 Dataset Statistics (Tokens):")
    display(df["tokens"].describe().round(0).to_frame().T)
    
    # Check Context Limits
    for model, limit in LIMITS.items():
        over = len(df[df["tokens"] > limit])
        if over > 0:
            print(f"🚨 {over} docs exceed {model} limit ({limit:,} tokens)")
else:
    print("❌ No data found.")

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 5))
    sns.histplot(df["tokens"], bins=30, kde=True, color="#3182ce")
    
    # Draw Limits
    colors = ['r', 'orange']
    for i, (model, limit) in enumerate(LIMITS.items()):
        plt.axvline(limit, color=colors[i % len(colors)], linestyle="--", label=f"{model} Limit")
    
    plt.title("Document Length Distribution")
    plt.xlabel("Token Count")
    plt.legend()
    plt.show()

In [ ]:
if not df.empty:
    avg_tokens = df["tokens"].mean()
    scenarios = [1000, 10000, 50000] # Number of documents
    
    print(f"💰 Cost Projection (Input Only) | Avg Doc: {int(avg_tokens):,} tokens")
    
    cost_data = []
    for num_docs in scenarios:
        total_m_tokens = (num_docs * avg_tokens) / 1_000_000
        row = {"Scenario": f"{num_docs:,} Docs ({total_m_tokens:.1f}M tok)"}
        
        for model, price in PRICING_1M.items():
            row[model] = f"${(total_m_tokens * price):.2f}"
        cost_data.append(row)
        
    cost_df = pd.DataFrame(cost_data)
    display(cost_df.style.hide(axis='index'))